### Imports

In [1]:
import json
import warnings
from pathlib import Path

import torch
import torch.nn as nn

from src.config import CONFIG
from src.datasets.audio_dataset import AudioDataset
from src.engine import benchmark_snn, validate_snn, train_one_epoch_snn, get_split_dataloaders
from src.models.snn_direct import SNNDirect
from src.preprocessing import get_snn_pipeline
from src.utils import plot_training_history, run_sweep_pareto

warnings.filterwarnings("ignore", category=UserWarning)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Constants

In [2]:
MODEL_NAME = SNNDirect.NAME

INPUT_DIR = Path('../../data/audio-mnist')
HYPERPARAMETERS_PATH = Path(f'../../hyperparameters/{MODEL_NAME}.json')
MODEL_PATH = Path(f'../../models/{MODEL_NAME}.pth')

### Device

In [3]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps' if torch.backends.mps.is_available() else
    'cpu'
)
print(f'Using device: {device}')

Using device: mps


### Hyperparameter Tuning

In [4]:
def objective(trial) -> tuple[float, int]:
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    beta = trial.suggest_float('beta_init', 0.5, 0.99)
    slope = trial.suggest_int('slope', 10, 50)
    timesteps = trial.suggest_categorical('timesteps', [5, 10, 15])

    dataset = AudioDataset(INPUT_DIR, get_snn_pipeline(CONFIG.audiomnist))
    train_dataloader, val_dataloader, _ = get_split_dataloaders(dataset)

    model = SNNDirect(beta_init=beta, slope=slope, timesteps=timesteps).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    val_acc = 0.0
    for epoch in range(CONFIG.hyperparameter_tuning.epochs):
        train_one_epoch_snn(device, model, criterion, optimiser, train_dataloader)
        val_loss, val_acc = validate_snn(device, model, criterion, val_dataloader)

    return val_acc, timesteps

if CONFIG.hyperparameter_tuning.should_run:
    run_sweep_pareto(objective, HYPERPARAMETERS_PATH)

### Training

In [5]:
if __name__ == '__main__':
    dataset = AudioDataset(INPUT_DIR, get_snn_pipeline(CONFIG.audiomnist))
    train_dataloader, val_dataloader, test_dataloader = get_split_dataloaders(dataset)

    # Load hyperparameters
    # NOTE: Make sure to change which set of parameters to use, by default take the lowest timestep
    hyperparameters = json.load(open(HYPERPARAMETERS_PATH, 'r'))[1]['params']
    print(f'Hyperparameters used: {hyperparameters}')
    print()

    # Train the model
    model = SNNDirect(beta_init=hyperparameters['beta_init'], slope=hyperparameters['slope'], timesteps=hyperparameters['timesteps']).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=hyperparameters['lr'])
    criterion = nn.CrossEntropyLoss()

    print(f'Training {model.NAME}...')
    best_acc = 0.0
    epochs_without_improvement = 0
    train_losses, train_accs = [], []
    val_losses, val_accs = [], []
    for epoch in range(CONFIG.epochs):
        print(f'[Epoch {epoch + 1}/{CONFIG.epochs}]')
        train_loss, train_acc = train_one_epoch_snn(device, model, criterion, optimizer, train_dataloader)
        train_losses.append(train_loss)
        train_accs.append(train_acc)

        val_loss, val_acc = validate_snn(device, model, criterion, val_dataloader)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement == CONFIG.patience:
            print(f'Stopping early at epoch {epoch + 1} (no improvement for {CONFIG.patience} epochs)')
            break

        print(f'Train Loss: {train_loss:.2f} | Train Accuracy: {train_acc:.2f}% | Val Loss: {val_loss:.2f} | Val Accuracy: {val_acc:.2f}%')
        print()
    print(f'Best model had an accuracy of {best_acc:.2f}%.')
    print(f'Running final test...', end='')
    model.load_state_dict(torch.load(MODEL_PATH))

    test_accuracy, acs, first_layer_macs = benchmark_snn(device, model, test_dataloader, direct_encoded=True)
    print(f'Test Accuracy: {test_accuracy:.2f}% | Average ACs per Inference: {acs} | First Layer MACs: {first_layer_macs}')

    plot_training_history(train_losses, train_accs, val_losses, val_accs)

Hyperparameters used: {'lr': 0.002166218735424637, 'beta_init': 0.5891520415207885, 'slope': 30, 'timesteps': 5}

Training snn...
[Epoch 1/50]


Validation:   0%|          | 0/12 [00:00<?, ?batches/s]Traceback (most recent call last):
  File "<string>", line 1, in <module>
    from multiprocessing.spawn import spawn_main; spawn_main(tracker_fd=94, pipe_handle=169)
                                                  ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/spawn.py", line 122, in spawn_main
    exitcode = _main(fd, parent_sentinel)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/multiprocessing/spawn.py", line 132, in _main
    self = reduction.pickle.load(from_parent)
  File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/pathlib/_local.py", line 505, in __new__
    def __new__(cls, *args, **kwargs):
    
KeyboardInterrupt


KeyboardInterrupt: 